# 02 Signal Research

## Objective

The EDA identified substantial common co-movement across cryptocurrencies,
with an average pairwise hourly-return correlation of approximately 0.63.

This notebook investigates whether asset-specific returns contain
cross-sectional predictive information after removing the common
crypto-market component.

Development sample: 2020–2023

In [2]:
# step 1 data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv(
    "crypto_hourly_clean_2020_2025.csv",
    parse_dates=["open_time"]
)

dev = df[
    (df["open_time"] >= "2020-01-01") &
    (df["open_time"] < "2024-01-01")
].copy()

return_panel = (
    dev
    .pivot(
        index="open_time",
        columns="symbol",
        values="ret_1h"
    )
    .sort_index()
)

print(return_panel.shape)
display(return_panel.head())

(35033, 8)


symbol,ADAUSDT,BNBUSDT,BTCUSDT,DOGEUSDT,ETHUSDT,LINKUSDT,LTCUSDT,XRPUSDT
open_time,,,,,,,,
2020-01-01 00:00:00+00:00,-0.002131,-0.001312,-0.002531,-0.002583,-0.002245,-0.001980,-0.000484,-0.002436
2020-01-01 01:00:00+00:00,0.006406,0.007402,0.005469,0.005577,0.013735,0.010942,0.008236,0.006390
2020-01-01 02:00:00+00:00,0.005456,0.003500,0.003683,0.004803,0.001607,0.009702,0.005526,0.002426
2020-01-01 03:00:00+00:00,-0.004221,-0.002224,-0.002463,-0.009610,-0.004968,-0.003999,-0.007646,-0.001081
2020-01-01 04:00:00+00:00,-0.001211,-0.001795,-0.001071,0.001642,0.000000,0.002008,0.001445,-0.001753


In [3]:
# step 2: calculate equal-weight crypto market return

market_return = return_panel.mean(axis=1)

print("Market observations:", len(market_return))
display(market_return.head())

print("\nMarket return statistics:")
display(market_return.describe())

Market observations: 35033


open_time
2020-01-01 00:00:00+00:00   -0.001963
2020-01-01 01:00:00+00:00    0.008020
2020-01-01 02:00:00+00:00    0.004588
2020-01-01 03:00:00+00:00   -0.004527
2020-01-01 04:00:00+00:00   -0.000092
dtype: float64


Market return statistics:


count    35018.000000
mean         0.000126
std          0.008960
min         -0.185252
25%         -0.003216
50%          0.000316
75%          0.003875
max          0.121130
dtype: float64

## Step 3: Estimate Rolling Market Beta

To separate common crypto-market movements from asset-specific returns,
a rolling market beta is estimated for each cryptocurrency.

Beta is estimated using the previous 30 days (720 hourly observations):

βᵢ,t = Cov(rᵢ, rₘ) / Var(rₘ)

where rᵢ is the hourly return of cryptocurrency i and rₘ is the
equal-weight crypto-market return.

The rolling approach allows market exposure to change over time.

In [5]:
# Step 3: Estimate 30-day rolling beta

beta_window = 24 * 30   # 30 days = 720 hours

rolling_beta = pd.DataFrame(
    index=return_panel.index,
    columns=return_panel.columns,
    dtype=float
)

# Rolling variance of crypto market return
market_var = market_return.rolling(
    window=beta_window,
    min_periods=beta_window
).var()

# Rolling beta for each cryptocurrency
for asset in return_panel.columns:

    rolling_cov = (
        return_panel[asset]
        .rolling(
            window=beta_window,
            min_periods=beta_window
        )
        .cov(market_return)
    )

    rolling_beta[asset] = rolling_cov / market_var

print("Rolling beta shape:", rolling_beta.shape)

display(
    rolling_beta
    .dropna(how="all")
    .head()
)

Rolling beta shape: (35033, 8)


symbol,ADAUSDT,BNBUSDT,BTCUSDT,DOGEUSDT,ETHUSDT,LINKUSDT,LTCUSDT,XRPUSDT
open_time,,,,,,,,
2020-01-30 23:00:00+00:00,1.151998,0.992259,0.743098,0.632351,0.981421,1.148859,1.358871,0.991143
2020-01-31 00:00:00+00:00,1.151313,0.992266,0.743217,0.631153,0.981396,1.150352,1.359009,0.991294
2020-01-31 01:00:00+00:00,1.151960,0.992443,0.743277,0.631006,0.979853,1.150087,1.359844,0.991530
2020-01-31 02:00:00+00:00,1.153944,0.992250,0.743106,0.631495,0.979954,1.148699,1.359696,0.990857
2020-01-31 03:00:00+00:00,1.152780,0.994831,0.742667,0.630546,0.979289,1.148857,1.360511,0.990518


In [6]:
# check if beta make sense
# Step 3.1: Inspect rolling beta estimates

beta_summary = rolling_beta.describe().T[
    ["mean", "std", "min", "25%", "50%", "75%", "max"]
]

display(beta_summary.round(3))

,mean,std,min,25%,50%,75%,max
symbol,,,,,,,
ADAUSDT,1.104,0.147,0.782,1.003,1.089,1.186,1.640
BNBUSDT,0.856,0.150,0.502,0.768,0.850,0.931,1.447
BTCUSDT,0.711,0.130,0.287,0.614,0.724,0.803,1.036
DOGEUSDT,1.090,0.312,0.536,0.915,1.066,1.228,2.713
ETHUSDT,0.924,0.146,0.532,0.847,0.925,1.017,1.260
LINKUSDT,1.250,0.191,0.758,1.146,1.202,1.314,1.966
LTCUSDT,1.066,0.139,0.738,0.981,1.026,1.149,1.492
XRPUSDT,0.999,0.250,0.692,0.840,0.954,1.039,2.399


## Step 4: Calculate Residual Returns

Using the estimated rolling beta, each cryptocurrency's hourly return
is decomposed into a common market component and an asset-specific
residual component.

Residual return is defined as:

Residual Return(i,t) = Return(i,t) - Beta(i,t) × Market Return(t)

The residual return represents the component of each cryptocurrency's
return that is not explained by contemporaneous movements in the
equal-weight crypto market.

In [8]:
# Step 4: Calculate residual returns

market_component = rolling_beta.mul(
    market_return,
    axis=0
)

residual_return = (
    return_panel - market_component
)

print("Residual return shape:", residual_return.shape)

display(
    residual_return
    .dropna(how="all")
    .head()
)

Residual return shape: (35033, 8)


symbol,ADAUSDT,BNBUSDT,BTCUSDT,DOGEUSDT,ETHUSDT,LINKUSDT,LTCUSDT,XRPUSDT
open_time,,,,,,,,
2020-01-30 23:00:00+00:00,-0.001924,0.002376,-0.000148,0.003372,-0.003779,0.009030,-0.006543,-0.002384
2020-01-31 00:00:00+00:00,0.005648,0.000320,-0.001639,0.008748,-0.000007,-0.011776,0.000299,-0.001593
2020-01-31 01:00:00+00:00,-0.001242,0.001123,-0.001119,-0.000251,-0.002604,0.002396,0.003713,-0.002016
2020-01-31 02:00:00+00:00,-0.009687,0.001511,0.000684,-0.003759,0.001355,0.004229,0.001233,0.004434
2020-01-31 03:00:00+00:00,-0.008517,0.014656,-0.003896,0.000824,-0.003887,0.000009,0.007113,-0.006302


In [9]:
# Step 4.1: Residual return summary

residual_summary = residual_return.describe().T[
    ["mean", "std", "min", "25%", "50%", "75%", "max"]
]

display(residual_summary.round(5))

,mean,std,min,25%,50%,75%,max
symbol,,,,,,,
ADAUSDT,-0.00002,0.00537,-0.09728,-0.00220,-0.00019,0.00181,0.06988
BNBUSDT,0.00002,0.00476,-0.07802,-0.00170,-0.00005,0.00160,0.09727
BTCUSDT,-0.00003,0.00325,-0.06456,-0.00131,-0.00003,0.00122,0.10582
DOGEUSDT,0.00008,0.01031,-0.18771,-0.00228,-0.00023,0.00182,0.53842
ETHUSDT,-0.00003,0.00361,-0.08854,-0.00150,-0.00003,0.00141,0.04061
LINKUSDT,-0.00000,0.00579,-0.09658,-0.00252,-0.00017,0.00221,0.10404
LTCUSDT,-0.00005,0.00498,-0.11066,-0.00211,-0.00015,0.00181,0.18999
XRPUSDT,0.00003,0.00658,-0.17427,-0.00223,-0.00013,0.00195,0.16469


In [10]:
# Step 4.2: Check whether market exposure has been reduced

comparison_rows = []

for asset in return_panel.columns:

    raw_corr = return_panel[asset].corr(market_return)
    residual_corr = residual_return[asset].corr(market_return)

    comparison_rows.append({
        "asset": asset,
        "raw_market_corr": raw_corr,
        "residual_market_corr": residual_corr
    })

corr_comparison = pd.DataFrame(comparison_rows)

display(corr_comparison.round(3))

,asset,raw_market_corr,residual_market_corr
0,ADAUSDT,0.838,-0.038
1,BNBUSDT,0.823,0.000
2,BTCUSDT,0.859,-0.019
3,DOGEUSDT,0.663,0.047
4,ETHUSDT,0.894,-0.022
5,LINKUSDT,0.855,-0.028
6,LTCUSDT,0.866,-0.032
7,XRPUSDT,0.770,0.027


- crypto return and equal-weight crypto market ：0.66−0.89
- but after residualisation ：-0.038∼0.047

In [14]:
# Step 5: Define development sample

dev_residual = residual_return.loc[
    (residual_return.index >= "2020-01-01") &
    (residual_return.index < "2024-01-01")
].copy()

dev_return = return_panel.loc[
    (return_panel.index >= "2020-01-01") &
    (return_panel.index < "2024-01-01")
].copy()

print(
    "Development period:",
    dev_residual.index.min(),
    "to",
    dev_residual.index.max()
)

print("Observations:", len(dev_residual))

Development period: 2020-01-01 00:00:00+00:00 to 2023-12-31 23:00:00+00:00
Observations: 35033


In [15]:
# Step 5.1: Construct residual momentum signals

formation_horizons = [24, 72, 168, 336]

residual_signals = {}

for h in formation_horizons:

    signal = dev_residual.rolling(
        window=h,
        min_periods=h
    ).sum()

    residual_signals[h] = signal

    print(
        f"{h}h residual momentum signal:",
        signal.dropna(how="all").shape
    )

24h residual momentum signal: (25764, 8)
72h residual momentum signal: (25284, 8)
168h residual momentum signal: (24324, 8)
336h residual momentum signal: (22760, 8)


In [16]:
# Step 6: Construct future compounded-return targets

target_horizons = [24, 72, 168]

future_returns = {}

for h in target_horizons:

    future_return = (
        (1 + dev_return)
        .rolling(window=h)
        .apply(np.prod, raw=True)
        .shift(-h)
        - 1
    )

    future_returns[h] = future_return

    print(
        f"Future {h}h return:",
        future_return.dropna(how="all").shape
    )

Future 24h return: (34649, 8)
Future 72h return: (33881, 8)
Future 168h return: (32480, 8)


In [17]:
# Step 6.1: Cross-sectional IC analysis

ic_results = []

for formation_h in formation_horizons:

    signal = residual_signals[formation_h]

    for target_h in target_horizons:

        target = future_returns[target_h]

        # Align signal and target
        common_index = signal.index.intersection(target.index)

        signal_aligned = signal.loc[common_index]
        target_aligned = target.loc[common_index]

        # Cross-sectional Spearman IC at each timestamp
        ic_series = signal_aligned.corrwith(
            target_aligned,
            axis=1,
            method="spearman"
        ).dropna()

        ic_results.append({
            "formation_h": formation_h,
            "target_h": target_h,
            "mean_IC": ic_series.mean(),
            "IC_std": ic_series.std(),
            "positive_IC_pct": (ic_series > 0).mean(),
            "n_obs": len(ic_series)
        })

ic_table = pd.DataFrame(ic_results)

display(ic_table.round(4))

,formation_h,target_h,mean_IC,IC_std,positive_IC_pct,n_obs
0,24,24,-0.0034,0.4063,0.4850,25524
1,24,72,-0.0002,0.4050,0.4911,25044
2,24,168,0.0113,0.4055,0.5050,24084
3,72,24,-0.0119,0.4090,0.4818,25044
4,72,72,-0.0044,0.3976,0.4879,24564
5,72,168,0.0031,0.3912,0.4869,23624
6,168,24,0.0017,0.4121,0.4924,24084
7,168,72,0.0048,0.4061,0.4979,23624
8,168,168,-0.0027,0.3979,0.4921,22760
9,336,24,-0.0044,0.4044,0.4846,22544


In [18]:
# Step 6.2: IC matrix

ic_matrix = ic_table.pivot(
    index="formation_h",
    columns="target_h",
    values="mean_IC"
)

ic_matrix.index = [f"{x}h signal" for x in ic_matrix.index]
ic_matrix.columns = [f"{x}h future" for x in ic_matrix.columns]

display(ic_matrix.round(4))

,24h future,72h future,168h future
24h signal,-0.0034,-0.0002,0.0113
72h signal,-0.0119,-0.0044,0.0031
168h signal,0.0017,0.0048,-0.0027
336h signal,-0.0044,0.0125,0.0212


# 336 h signal and 168h future positive ic

- past 14 days: residual performance good ，next 7 day, momentum

-  Residual Momentum

In [20]:
# Step 6.3: Year-by-year IC robustness
# Focus on the strongest candidate: 336h signal -> 168h future return

signal = residual_signals[336]
target = future_returns[168]

common_index = signal.index.intersection(target.index)

signal = signal.loc[common_index]
target = target.loc[common_index]

yearly_ic_results = []

for year in [2020, 2021, 2022, 2023]:

    mask = signal.index.year == year

    signal_year = signal.loc[mask]
    target_year = target.loc[mask]

    ic_series = signal_year.corrwith(
        target_year,
        axis=1,
        method="spearman"
    ).dropna()

    yearly_ic_results.append({
        "year": year,
        "mean_IC": ic_series.mean(),
        "median_IC": ic_series.median(),
        "positive_IC_pct": (ic_series > 0).mean(),
        "n_obs": len(ic_series)
    })

yearly_ic = pd.DataFrame(yearly_ic_results)

display(yearly_ic.round(4))

,year,mean_IC,median_IC,positive_IC_pct,n_obs
0,2020,0.1807,0.1905,0.7069,2825
1,2021,-0.0615,-0.0952,0.4176,2603
2,2022,0.0136,0.0238,0.5031,8760
3,2023,-0.0017,0.0000,0.4910,7368


In [21]:
# Step 6.4: Year-by-year IC for all signal-target combinations

yearly_ic_all = []

for formation_h in formation_horizons:

    signal = residual_signals[formation_h]

    for target_h in target_horizons:

        target = future_returns[target_h]

        common_index = signal.index.intersection(target.index)

        signal_aligned = signal.loc[common_index]
        target_aligned = target.loc[common_index]

        # Overall IC
        overall_ic_series = signal_aligned.corrwith(
            target_aligned,
            axis=1,
            method="spearman"
        ).dropna()

        overall_mean_ic = overall_ic_series.mean()

        # Year-by-year IC
        row = {
            "formation_h": formation_h,
            "target_h": target_h,
            "overall_IC": overall_mean_ic
        }

        for year in [2020, 2021, 2022, 2023]:

            mask = signal_aligned.index.year == year

            signal_year = signal_aligned.loc[mask]
            target_year = target_aligned.loc[mask]

            ic_series = signal_year.corrwith(
                target_year,
                axis=1,
                method="spearman"
            ).dropna()

            row[f"IC_{year}"] = ic_series.mean()

        yearly_ic_all.append(row)

yearly_ic_table = pd.DataFrame(yearly_ic_all)

display(
    yearly_ic_table.round(4)
)

,formation_h,target_h,overall_IC,IC_2020,IC_2021,IC_2022,IC_2023
0,24,24,-0.0034,-0.0230,0.0212,0.0147,-0.0261
1,24,72,-0.0002,0.0433,0.0084,0.0063,-0.0350
2,24,168,0.0113,0.0196,-0.0048,0.0300,-0.0060
3,72,24,-0.0119,0.0413,-0.0095,-0.0047,-0.0493
4,72,72,-0.0044,0.0972,-0.0113,-0.0239,-0.0307
5,72,168,0.0031,0.0234,-0.0365,0.0307,-0.0196
6,168,24,0.0017,0.0201,-0.0318,0.0252,-0.0175
7,168,72,0.0048,0.0376,-0.0751,0.0256,0.0022
8,168,168,-0.0027,-0.0269,-0.0734,0.0483,-0.0217
9,336,24,-0.0044,0.0528,-0.0573,-0.0058,-0.0073


# conclusion
- Residual momentum exists in the pooled development sample, but the signal is not stable across market regimes. The strongest overall specification, 336h formation / 168h target, is largely driven by strong momentum in 2020 and reverses in 2021.

# 
Candidate A：Residual Momentum
- formation = 336h
- holding / target = 168h
- Long strongest residual winners
- Short weakest residual losers
#

#Candidate B：Residual Reversal
- formation = 72h
- holding / target = 24h
- Long weakest residual losers
- Short strongest residual winners